# Survey Responses on Behavioral Intention and Influencing Factors among Philippine Accounting Students Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.6dhf-2sn3/fair2.json'

# Load the dataset with mlcroissant
dataset = mlc.Dataset(croissant_url)

# Print metadata summary
meta = dataset.metadata
print(f"{meta.name}: {meta.description}\n")
print(f"Published: {meta.datePublished}\nVersion: {meta.version}")
print(f"Identifier: {meta.identifier}")
print(f"License: {meta.license}")


## 2. Data Overview
Review available record sets and their `@id`s. Each record set groups related data, while each field/column is uniquely identified by `@id`.

In [ ]:
from pprint import pprint

# List Record Sets (`cr:RecordSet`) by @id (using the metadata object)
record_sets = dataset.metadata.record_sets
print("Record Sets (@id):")
for rs in record_sets:
    print(f"  - {rs['@id']} : {rs['name'] if 'name' in rs else 'No name'}")
if not record_sets:
    print("(No explicit recordSet listed in root metadata. Attempting to infer from schema/records interface.)")

# Use the dataset interface to enumerate record set ids (if possible)
# mlcroissant builds record set ids from the schema automatically; list available record set ids
try:
    discovered_record_sets = dataset.record_set_ids
    print("\nRecord sets discovered in schema via dataset.record_set_ids:")
    for r in discovered_record_sets:
        print("  -", r)
except Exception as e:
    print("Could not enumerate record_set_ids automatically in this version of mlcroissant.")

# Preview field/column @id's for the main record set
# Let's select the first discovered record set if available
main_record_set_id = None
if record_sets:
    main_record_set_id = record_sets[0]['@id']
elif 'discovered_record_sets' in locals() and discovered_record_sets:
    main_record_set_id = discovered_record_sets[0]

if main_record_set_id is not None:
    print(f"\nField IDs in record set {main_record_set_id}:")
    fields = dataset.metadata.fields(main_record_set_id)
    for field in fields:
        fname = field.get('name', field.get('@id'))
        print(f"  - {field['@id']} (name: {fname})")
else:
    print("No record set id found.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the Record Set and field `@id`s from the overview above.

In [ ]:
# Use all discovered record sets (if available)
record_set_ids = []
if record_sets:
    record_set_ids = [rs['@id'] for rs in record_sets]
elif 'discovered_record_sets' in locals():
    record_set_ids = discovered_record_sets

print(f"Loading these record sets: {record_set_ids}")
dataframes = {}
for record_set_id in record_set_ids:
    # Each record is a dict of field @id: value
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

if main_record_set_id is not None:
    display_fields = dataframes[main_record_set_id].columns.tolist()
    print(f"\nColumns for record set {main_record_set_id}:\n", display_fields)
    dataframes[main_record_set_id].head()
else:
    print("No main record set id available to print preview.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalization, and grouping. Variables are referenced by their `@id` per the schema.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Choose a numeric field (by @id) to analyze. Update as appropriate for your dataset.
# We'll auto-select a likely numeric field if present (e.g. columns that look like Likert-scale answers)
df = dataframes[main_record_set_id].copy()
candidate_numeric_fields = []
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]) or df[col].dropna().astype(str).str.isnumeric().any():
        candidate_numeric_fields.append(col)

if candidate_numeric_fields:
    numeric_field = candidate_numeric_fields[0]  # Choose first numeric column
else:
    numeric_field = df.columns[0]  # Fallback

# Ensure dtype is numeric
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

# Show summary
print(f"Analyzing numeric field: {numeric_field}")

# Filter: keep records with value > threshold
threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records where {numeric_field} > {threshold:.2f}:")
print(filtered_df[[numeric_field]].head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a field. Choose next available field that's categorical, if present.
group_field = None
for col in df.columns:
    if col != numeric_field and df[col].nunique() < 10:
        group_field = col
        break

if group_field is not None:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"\nGrouped filtered data by {group_field}:\n{grouped_df.head()}")
else:
    print("\nNo categorical field with few unique values found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

# Histogram of the selected numeric field
plt.figure(figsize=(7, 4))
df[numeric_field].hist(bins=8, color='cornflowerblue', edgecolor='black')
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.show()

# Boxplot by group (if a group_field is identified)
if group_field is not None:
    plt.figure(figsize=(7, 4))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f'{numeric_field} distribution by {group_field}')
    plt.xticks(rotation=35)
    plt.show()

## 6. Conclusion
This exploratory notebook provided an overview and basic analysis of the FAIR^2 survey dataset on behavioral intention among Philippine accounting students.

- We loaded the dataset using the Croissant schema and referenced fields/record sets by their `@id`.
- Basic data cleaning, normalization, and grouping were demonstrated for Likert-scale or numeric responses.
- Visualizations such as histograms and boxplots can be used to further explore variable distributions and group differences.

For more detailed analysis, consult the Croissant schema (fields' `@id`/meaning), and tailor analysis to specific research questions using the schema's structure.